In [7]:

import wandb
import numpy as np
import joblib
from SRC.helper_functions.preprocessing import processor, prep_x_for_tf, prep_y_for_tf
import polars as pl

In [ ]:
'''list of all sites    "sites": [
        "06936530", "06930000", "06900050", "06925250", "06929900",
        "06897500", "06901250", "06893970", "06928420", "06899700",
        "06935997", "06893620", "06906150", "06918440", "06932000",
        "06894000", "06935955", "06923250", "06900640", "06909500",
        "06901500", "06907700", "06893830", "06928380", "06896000",
        "06935770", "06919500", "06893390", "06920520", "06933500",
        "06928300", "06921600", "06900800", "06906300", "06897000",
        "06918740", "06909950", "06936475", "06908000", "06935850",
        "06902995", "06893820", "06930060", "06820500", "06899500",
        "06923940", "06935755", "06904500", "06910230", "06904650",
        "06927000", "06917060", "06921720", "06918460", "06921200",
        "06905500", "06918493", "06928000", "06899900", "06917630",
        "06901205", "06923950", "06894200", "06928330", "06906800",
        "06928359", "06893940", "06927240", "06893150", "06893557",
        "06917560", "06821080", "06921070", "06935830", "06935980",
        "06902000", "06896400", "06918060", "06926290", "06921590",
        "06893578", "06930015", "06928320", "06893750", "06895000",
        "06910750", "06934000", "06935890", "06821150", "06893500",
        "06906000", "06896900",
    ]
'''
config = {
    "input_cols": [
        "latitude",
        "longitude",
        "streamflow_cfs_mean",
        "gage_height_ft_mean",
        "precipitation_mm",
        "temperature_c",
        "specific_humidity_kgkg",
    ], 
    "target": "streamflow_cfs_mean",
    "train_split": 0.8,
    "val_split": 0.9,
    "n_rows": 3650,
    "file_path": "flood-dataset-missouri",
    "file_name": "flood_model_missouri",
    "table": "wandb.flood_model_missouri",
    "lag_window": 3,
    "sites": [
        "06936530", "06930000", "06900050", "06925250", "06929900",
        "06897500", "06901250", "06893970", "06928420", "06899700",
        "06935997", "06893620", "06906150", "06918440", "06932000",
        "06894000", "06935955", "06923250", "06900640", "06909500",
        "06901500", "06907700", "06893830", "06928380", "06896000",
        "06935770", "06919500", "06893390", "06920520", "06933500",
        "06928300", "06921600", "06900800", "06906300", "06897000",
        "06918740", "06909950", "06936475", "06908000", "06935850",
        "06902995", "06893820", "06930060", "06820500", "06899500",
        "06923940", "06935755", "06904500", "06910230", "06904650",
        "06927000", "06917060", "06921720", "06918460", "06921200",
        "06905500", "06918493", "06928000", "06899900", "06917630",
        "06901205", "06923950", "06894200", "06928330", "06906800",
        "06928359", "06893940", "06927240", "06893150", "06893557",
        "06917560", "06821080", "06921070", "06935830", "06935980",
        "06902000", "06896400", "06918060", "06926290", "06921590",
        "06893578", "06930015", "06928320", "06893750", "06895000",
        "06910750", "06934000", "06935890", "06821150", "06893500",
        "06906000", "06896900",
    ]
}

pcr = processor(config) 
pcr.pull_duckdb()
(
    train_X_scaled,
    val_X_scaled,
    test_X_scaled,
    train_y_scaled,
    val_y_scaled,
    test_y_scaled,
) = pcr.return_outputs()
train_X_scaled

[shape: (748, 19)
 ┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
 │ site_id  ┆ observati ┆ streamflo ┆ gage_heig ┆ … ┆ temperatu ┆ temperatu ┆ specific_ ┆ specific_ │
 │ ---      ┆ on_hour   ┆ w_cfs_mea ┆ ht_ft_mea ┆   ┆ re_c1     ┆ re_c2     ┆ humidity_ ┆ humidity_ │
 │ str      ┆ ---       ┆ n         ┆ n         ┆   ┆ ---       ┆ ---       ┆ kgkg1     ┆ kgkg2     │
 │          ┆ datetime[ ┆ ---       ┆ ---       ┆   ┆ f64       ┆ f64       ┆ ---       ┆ ---       │
 │          ┆ μs, Ameri ┆ f64       ┆ f64       ┆   ┆           ┆           ┆ f64       ┆ f64       │
 │          ┆ ca/Chicag ┆           ┆           ┆   ┆           ┆           ┆           ┆           │
 │          ┆ o]        ┆           ┆           ┆   ┆           ┆           ┆           ┆           │
 ╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
 │ 06936530 ┆ 2007-10-2 ┆ -0.214167 ┆ -1.625424 ┆ … ┆ NaN       

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
import numpy as np


# infer timesteps (lag window) from config
timesteps = config.get('lag_window', 3)
drop_col = ['latitude', 'longitude', 'site_id', 'observation_hour']

# Build combined 3D arrays for X features
X_train = prep_x_for_tf(train_X_scaled, drop_col, timesteps) 
X_val = prep_x_for_tf(val_X_scaled, drop_col, timesteps)

# Build aligned y arrays (one column per site)
y_train = prep_y_for_tf(train_y_scaled, timesteps, X_train.shape[0])
y_val = prep_y_for_tf(val_y_scaled, timesteps, X_val.shape[0])


model = Sequential([
    GRU(64, activation='tanh', return_sequences=True,),
    GRU(32, activation='tanh'),
    Dense(len(config['sites']), activation='linear')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:

model.fit(X_train, y_train, epochs=10, batch_size=1, validation_data=(X_val, y_val))

Epoch 1/10
745/745 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.5045 - mae: 0.2196 - val_loss: 0.0748 - val_mae: 0.1359
Epoch 2/10
745/745 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2036 - mae: 0.1609 - val_loss: 0.0326 - val_mae: 0.1032
Epoch 3/10
745/745 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1158 - mae: 0.1491 - val_loss: 0.0528 - val_mae: 0.1237
Epoch 4/10
745/745 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0767 - mae: 0.1209 - val_loss: 0.0158 - val_mae: 0.0800
Epoch 5/10
745/745 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0614 - mae: 0.1109 - val_loss: 0.0098 - val_mae: 0.0697
Epoch 6/10
745/745 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0832 - mae: 0.1133 - val_loss: 0.0074 - val_mae: 0.0579
Epoch 7/10
745/745 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0272 - mae: 0.0847 - val_loss: 0.0076 - val_mae: 0.0579
Epoch 8/10
745/745 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0292 - mae: 0.0851 - val_loss: 0.0120 - val_mae: 0.0594
Epoch 9/10
745/745 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - lo

In [11]:
print('X_val:', X_val.shape)
print('y_train:', y_train.shape)
print('y_val:', y_val[0])

X_val: (90, 3, 5)
y_train: (745, 1)
y_val: [-0.03197374]
